In [1]:
# Mounting Drive
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/mzazicare-ml'
DATA = f'{BASE}/data'
MODELS = f'{BASE}/models'

import os
os.makedirs(DATA, exist_ok=True)
os.makedirs(MODELS, exist_ok=True)
print("Drive mounted and paths ready")

Mounted at /content/drive
Drive mounted and paths ready


1. Loading the data sample

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv(f'{DATA}/rba_sample.csv')
print(f"Loaded: {df.shape}")
print(df.columns.tolist())

Loaded: (1563463, 16)
['index', 'Login Timestamp', 'User ID', 'Round-Trip Time [ms]', 'IP Address', 'Country', 'Region', 'City', 'ASN', 'User Agent String', 'Browser Name and Version', 'OS Name and Version', 'Device Type', 'Login Successful', 'Is Attack IP', 'Is Account Takeover']


2.
Parsing timestamps, extracting time features and building per-user login time baseline using personalized mean and standard deviation

In [3]:
df['login_time'] = pd.to_datetime(df['Login Timestamp'])
df['login_hour'] = df['login_time'].dt.hour
df['login_day_of_week'] = df['login_time'].dt.dayofweek  # 0=Monday, 6=Sunday
df['login_month'] = df['login_time'].dt.month

# For each user, compute their typical login hour/
user_time_stats = df.groupby('User ID')['login_hour'].agg(['mean', 'std']).reset_index()
user_time_stats.columns = ['User ID', 'user_mean_hour', 'user_std_hour']

# Fill NaN std (users with only one login) with 0
user_time_stats['user_std_hour'] = user_time_stats['user_std_hour'].fillna(0)

df = df.merge(user_time_stats, on='User ID', how='left')

# Time deviation — how far this login is from the user's usual hour
df['time_deviation'] = abs(df['login_hour'] - df['user_mean_hour'])

print("Time features done:")
print(df[['login_hour', 'time_deviation']].describe())

Time features done:
         login_hour  time_deviation
count  1.563463e+06    1.563463e+06
mean   1.222946e+01    2.808654e+00
std    5.679111e+00    3.304531e+00
min    0.000000e+00    0.000000e+00
25%    8.000000e+00    0.000000e+00
50%    1.200000e+01    1.500000e+00
75%    1.700000e+01    4.796820e+00
max    2.300000e+01    1.941667e+01


3. Building per-user location baselines

In [4]:
# Identify common country per user
user_country = df.groupby('User ID')['Country'].agg(
    lambda x: x.mode()[0] if not x.mode().empty else 'Unknown'
).reset_index()
user_country.columns = ['User ID', 'usual_country']

# Identify most common city per user
user_city = df.groupby('User ID')['City'].agg(
    lambda x: x.mode()[0] if not x.mode().empty else 'Unknown'
).reset_index()
user_city.columns = ['User ID', 'usual_city']

# Merge both into df
df = df.merge(user_country, on='User ID', how='left')
df = df.merge(user_city, on='User ID', how='left')

# Feature flags
df['is_known_country'] = (df['Country'] == df['usual_country']).astype(int)
df['is_known_city'] = (df['City'] == df['usual_city']).astype(int)
df['is_foreign_login'] = (df['Country'] != df['usual_country']).astype(int)
df['location_anomaly_score'] = (
    df['is_foreign_login'] + (1 - df['is_known_city'])
)

print("Location features added:")
print(df[['is_known_country', 'is_known_city',
          'is_foreign_login', 'location_anomaly_score']].describe())
print("\nLocation anomaly score distribution:")
print(df['location_anomaly_score'].value_counts().sort_index())

Location features added:
       is_known_country  is_known_city  is_foreign_login  \
count      1.563463e+06   1.563463e+06      1.563463e+06   
mean       6.973654e-01   7.084127e-01      3.026346e-01   
std        4.593986e-01   4.544934e-01      4.593986e-01   
min        0.000000e+00   0.000000e+00      0.000000e+00   
25%        0.000000e+00   0.000000e+00      0.000000e+00   
50%        1.000000e+00   1.000000e+00      0.000000e+00   
75%        1.000000e+00   1.000000e+00      1.000000e+00   
max        1.000000e+00   1.000000e+00      1.000000e+00   

       location_anomaly_score  
count            1.563463e+06  
mean             5.942219e-01  
std              8.150522e-01  
min              0.000000e+00  
25%              0.000000e+00  
50%              0.000000e+00  
75%              1.000000e+00  
max              2.000000e+00  

Location anomaly score distribution:
location_anomaly_score
0    965238
1    267406
2    330819
Name: count, dtype: int64



4. Measuring login velocity, which is the number of login attempts per hour for each user

In [5]:
df = df.sort_values(['User ID', 'login_time']).reset_index(drop=True)
#Count logins per user in a rolling 1-hour window
df['login_timestamp_unix'] = df['login_time'].astype(np.int64) // 10**9

results = []
total_users = df['User ID'].nunique()
print(f"Computing velocity for {total_users:,} users...")

# Count logins within the last 3600 seconds (1 hour)
for i, (user_id, group) in enumerate(df.groupby('User ID', sort=False)):
    times = group['login_timestamp_unix'].values
    lower_bounds = np.searchsorted(times, times - 3600, side='left')
    current_pos = np.arange(len(times)) + 1
    group = group.copy()
    group['login_velocity'] = current_pos - lower_bounds
    results.append(group)
    if i % 50000 == 0:
        print(f"  Progress: {i:,} / {total_users:,} users")

df = pd.concat(results, ignore_index=True)

# Verify User ID retention
assert 'User ID' in df.columns, "User ID lost — check groupby"
print(f"Done. Shape: {df.shape}")
print(f"User ID present: {'User ID' in df.columns}")
print(f"Sample User IDs: {df['User ID'].head(3).tolist()}")
print(df['login_velocity'].describe())

Computing velocity for 672,724 users...
  Progress: 0 / 672,724 users
  Progress: 50,000 / 672,724 users
  Progress: 100,000 / 672,724 users
  Progress: 150,000 / 672,724 users
  Progress: 200,000 / 672,724 users
  Progress: 250,000 / 672,724 users
  Progress: 300,000 / 672,724 users
  Progress: 350,000 / 672,724 users
  Progress: 400,000 / 672,724 users
  Progress: 450,000 / 672,724 users
  Progress: 500,000 / 672,724 users
  Progress: 550,000 / 672,724 users
  Progress: 600,000 / 672,724 users
  Progress: 650,000 / 672,724 users
Done. Shape: (1563463, 31)
User ID present: True
Sample User IDs: [-9223369357534132497, -9223346519290912447, -9223320431696609944]
count    1.563463e+06
mean     4.514259e+01
std      6.111028e+01
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      7.700000e+01
max      4.960000e+02
Name: login_velocity, dtype: float64


5. Identifying login device (device fingerprint)

In [6]:
# Use User Agent as a proxy for device fingerprint
# Flag 1 if this user agent has been seen before for this user, 0 if new
user_agents_seen = df.groupby('User ID')['User Agent String'].apply(
    lambda x: x.duplicated(keep='first')
).reset_index(level=0, drop=True)

df['is_known_device'] = user_agents_seen.astype(int)

print("Known device distribution:")
print(df['is_known_device'].value_counts())

Known device distribution:
is_known_device
0    879859
1    683604
Name: count, dtype: int64


6. Saving checkpoint to Drive

In [7]:
assert 'User ID' in df.columns
assert 'login_velocity' in df.columns
assert 'is_known_device' in df.columns

df.to_parquet(f'{DATA}/checkpoint_all_features.parquet', index=False)
print(f"Checkpoint saved. Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Checkpoint saved. Shape: (1563463, 32)
Columns: ['index', 'Login Timestamp', 'User ID', 'Round-Trip Time [ms]', 'IP Address', 'Country', 'Region', 'City', 'ASN', 'User Agent String', 'Browser Name and Version', 'OS Name and Version', 'Device Type', 'Login Successful', 'Is Attack IP', 'Is Account Takeover', 'login_time', 'login_hour', 'login_day_of_week', 'login_month', 'user_mean_hour', 'user_std_hour', 'time_deviation', 'usual_country', 'usual_city', 'is_known_country', 'is_known_city', 'is_foreign_login', 'location_anomaly_score', 'login_timestamp_unix', 'login_velocity', 'is_known_device']


7. Implementing final feature set

In [11]:
features = [
    'login_hour',
    'login_day_of_week',
    'time_deviation',
    'is_known_country',
    'is_known_city',
    'is_foreign_login',
    'location_anomaly_score',
    'is_known_device',
    'login_velocity',
]

labels = ['Is Attack IP', 'Login Successful', 'Is Account Takeover']

df_features = df[features + labels].copy()
df_features = df_features.dropna(subset=features)

for col in labels:
    df_features[col] = df_features[col].astype(int)

print(f"Final feature dataset: {df_features.shape}")
print(f"Attack IPs: {df_features['Is Attack IP'].sum():,}")
print(f"Account takeovers: {df_features['Is Account Takeover'].sum():,}")
print(df_features[features].describe().round(4))

Final feature dataset: (1563463, 12)
Attack IPs: 155,004
Account takeovers: 7
         login_hour  login_day_of_week  time_deviation  is_known_country  \
count  1.563463e+06       1.563463e+06    1.563463e+06      1.563463e+06   
mean   1.222950e+01       2.889600e+00    2.808700e+00      6.974000e-01   
std    5.679100e+00       1.982700e+00    3.304500e+00      4.594000e-01   
min    0.000000e+00       0.000000e+00    0.000000e+00      0.000000e+00   
25%    8.000000e+00       1.000000e+00    0.000000e+00      0.000000e+00   
50%    1.200000e+01       3.000000e+00    1.500000e+00      1.000000e+00   
75%    1.700000e+01       5.000000e+00    4.796800e+00      1.000000e+00   
max    2.300000e+01       6.000000e+00    1.941670e+01      1.000000e+00   

       is_known_city  is_foreign_login  location_anomaly_score  \
count   1.563463e+06      1.563463e+06            1.563463e+06   
mean    7.084000e-01      3.026000e-01            5.942000e-01   
std     4.545000e-01      4.594000e-01 

A. Loading the checkpoint features

In [8]:
df_full = pd.read_parquet(f'{DATA}/checkpoint_all_features.parquet')
print(f"Loaded: {df_full.shape}")
print(df_full.columns.tolist())

Loaded: (1563463, 32)
['index', 'Login Timestamp', 'User ID', 'Round-Trip Time [ms]', 'IP Address', 'Country', 'Region', 'City', 'ASN', 'User Agent String', 'Browser Name and Version', 'OS Name and Version', 'Device Type', 'Login Successful', 'Is Attack IP', 'Is Account Takeover', 'login_time', 'login_hour', 'login_day_of_week', 'login_month', 'user_mean_hour', 'user_std_hour', 'time_deviation', 'usual_country', 'usual_city', 'is_known_country', 'is_known_city', 'is_foreign_login', 'location_anomaly_score', 'login_timestamp_unix', 'login_velocity', 'is_known_device']


B. Engineering new behavioural features

In [10]:
df_full = df_full.sort_values(['User ID', 'login_time']).reset_index(drop=True)

# 1. Hours since this user's previous login
df_full['prev_login_time'] = df_full.groupby('User ID')['login_time'].shift(1)
df_full['hours_since_last_login'] = (
    (df_full['login_time'] - df_full['prev_login_time']).dt.total_seconds() / 3600
)
# First-ever login per user has no previous login — fill with median
median_gap = df_full['hours_since_last_login'].median()
df_full['hours_since_last_login'] = df_full['hours_since_last_login'].fillna(median_gap)

# 2. New ASN (network provider) never seen before for this user
df_full['asn_seen_before'] = df_full.groupby('User ID')['ASN'].apply(
    lambda x: x.duplicated(keep='first')
).reset_index(level=0, drop=True)
df_full['is_new_asn'] = (~df_full['asn_seen_before']).astype(int)

# 3. Browser/OS combination mismatch from user's usual pattern
df_full['browser_os'] = (
    df_full['Browser Name and Version'].astype(str) + '_' +
    df_full['OS Name and Version'].astype(str)
)
df_full['browser_os_seen_before'] = df_full.groupby('User ID')['browser_os'].apply(
    lambda x: x.duplicated(keep='first')
).reset_index(level=0, drop=True)
df_full['is_new_browser_os'] = (~df_full['browser_os_seen_before']).astype(int)

# 4. Device type mismatch
df_full['device_type_seen_before'] = df_full.groupby('User ID')['Device Type'].apply(
    lambda x: x.duplicated(keep='first')
).reset_index(level=0, drop=True)
df_full['is_new_device_type'] = (~df_full['device_type_seen_before']).astype(int)

print("New features engineered:")
print(df_full[['hours_since_last_login', 'is_new_asn',
                'is_new_browser_os', 'is_new_device_type']].describe())

New features engineered:
       hours_since_last_login    is_new_asn  is_new_browser_os  \
count            1.563463e+06  1.563463e+06       1.563463e+06   
mean             1.874117e+02  4.556987e-01       5.084118e-01   
std              8.269200e+02  4.980337e-01       4.999294e-01   
min              5.555556e-07  0.000000e+00       0.000000e+00   
25%              9.199444e-03  0.000000e+00       0.000000e+00   
50%              1.172917e-02  0.000000e+00       1.000000e+00   
75%              1.502042e-02  1.000000e+00       1.000000e+00   
max              9.363013e+03  1.000000e+00       1.000000e+00   

       is_new_device_type  
count        1.563463e+06  
mean         4.515713e-01  
std          4.976493e-01  
min          0.000000e+00  
25%          0.000000e+00  
50%          0.000000e+00  
75%          1.000000e+00  
max          1.000000e+00  


C. Redoing feature comparison with new device type, new browser OS and new network features

In [12]:
new_features = [
    'login_hour', 'login_day_of_week', 'time_deviation',
    'is_known_country', 'is_known_city', 'is_foreign_login',
    'location_anomaly_score', 'is_known_device', 'login_velocity',
    'hours_since_last_login', 'is_new_asn', 'is_new_browser_os', 'is_new_device_type'
]

comparison_df = df_full[new_features + ['Is Attack IP']].dropna()

print("Mean feature values — Normal vs Attack (including new features):")
comparison = comparison_df.groupby('Is Attack IP')[new_features].mean().T
comparison.columns = ['Normal', 'Attack']
comparison['difference'] = abs(comparison['Attack'] - comparison['Normal'])
comparison = comparison.sort_values('difference', ascending=False)
print(comparison.round(4))

Mean feature values — Normal vs Attack (including new features):
                          Normal   Attack  difference
hours_since_last_login  201.4616  59.7452    141.7165
login_velocity           44.6339  49.7649      5.1310
login_hour               12.3756  10.9012      1.4744
location_anomaly_score    0.6285   0.2829      0.3456
is_known_country          0.6791   0.8634      0.1843
is_foreign_login          0.3209   0.1366      0.1843
is_known_city             0.6924   0.8537      0.1613
login_day_of_week         2.8790   2.9863      0.1073
is_new_browser_os         0.5164   0.4357      0.0807
is_known_device           0.4325   0.4804      0.0479
is_new_device_type        0.4558   0.4131      0.0427
is_new_asn                0.4595   0.4211      0.0384
time_deviation            2.8086   2.8094      0.0008


8. Saving feature dataset to Drive

In [ ]:
df_features.to_parquet(f'{DATA}/rba_features.parquet', index=False)
df_features.to_csv(f'{DATA}/rba_features.csv', index=False)
print(f"Feature dataset saved to Drive")
print(f"Shape: {df_features.shape}")

Feature dataset saved to Drive
Shape: (1563463, 12)


9. Saving the expanded feature dataset to Drive

In [14]:
# Saving the expanded feature set as a new file
final_features = new_features + ['Is Attack IP', 'Login Successful', 'Is Account Takeover']
df_features_v2 = df_full[final_features].dropna(subset=new_features)

df_features_v2.to_parquet(f'{DATA}/rba_features_v2.parquet', index=False)
print(f"Saved expanded feature set: {df_features_v2.shape}")
print(df_features_v2.columns.tolist())

Saved expanded feature set: (1563463, 16)
['login_hour', 'login_day_of_week', 'time_deviation', 'is_known_country', 'is_known_city', 'is_foreign_login', 'location_anomaly_score', 'is_known_device', 'login_velocity', 'hours_since_last_login', 'is_new_asn', 'is_new_browser_os', 'is_new_device_type', 'Is Attack IP', 'Login Successful', 'Is Account Takeover']
